# Synthesis and Capture

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from acadia.system import Acadia
from acadia.arrays import Waveform

No module named 'pyxrfdc'
No module named 'pyxrfclk'


## The actual meat of the program

In [3]:
acadia = Acadia()

pulse_channel = acadia.DAC(1)

pulse = Waveform(pulse_channel, 
                 data=0.99*np.ones(1000, dtype=np.complex64), 
                 region=pulse_channel)

capture_channel = acadia.ADC(1)
capture_data = Waveform(capture_channel, 
                        length_seconds=5000e-9, 
                        region=acadia.PLDDR0Array)


# Create a sequence for the sequencer
def sequence(a):
    with a.synchronizer():
        a.generate(pulse_channel, pulse)
        a.capture(capture_channel, capture_data)


# Attach to the hardware
acadia.attach()

# Now that we've attached, we can load the data into memory
pulse.flush()

# Configure various channel parameters
pulse_channel.set_nyquist_zone(2)
pulse_channel.configure_nco(frequency=2000e6)
pulse_channel.set_vop(20000)

capture_channel.set_nyquist_zone(2)
capture_channel.set_dsa(0)

# Now actually run the sequencer
acadia.run(sequence)

In [ ]:
# Get the sample data from capture memory
capture_data.unflush()
times = capture_data.axis()*1e6

# Plot the captured signal
fig,ax = plt.subplots()
ax.plot(times, np.real(capture_data.memory), label="Re")
ax.plot(times, np.imag(capture_data.memory), label="Im")
ax.set_xlabel("Time (us)")
ax.set_ylabel("Amplitude (\%FS)")
ax.grid()
ax.legend()